# 02: Knowledge Graphs & Graph Convolutional Networks (GCN)

**Track 03: Graph Data Science & Network Analysis** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Construct RDF-style knowledge triples (Subject-Predicate-Object), query entity relations, and implement a Graph Convolutional Network (GCN) layer from scratch in PyTorch.


## 1. Knowledge Graph Representation (Triples)
Representing entity relationships and executing relational SPARQL/Cypher style lookups.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

triples = [
    ("Tensorbox", "IS_A", "ML_Workstation"),
    ("Tensorbox", "SUPPORTS", "PyTorch"),
    ("Tensorbox", "SUPPORTS", "XGBoost"),
    ("PyTorch", "USED_FOR", "Deep_Learning"),
    ("XGBoost", "USED_FOR", "Tabular_ML"),
    ("Deep_Learning", "SUBFIELD_OF", "Artificial_Intelligence"),
    ("Tabular_ML", "SUBFIELD_OF", "Artificial_Intelligence")
]

kg_df = pd.DataFrame(triples, columns=["Subject", "Predicate", "Object"])
print("Knowledge Graph Triples:")
print(kg_df)

# Querying all technologies used for AI
ai_techs = kg_df[kg_df["Predicate"] == "USED_FOR"]["Subject"].tolist()
print(f"\nTechnologies contributing to AI: {ai_techs}")

## 2. Graph Convolutional Network (GCN) Layer from Scratch
Implementing the spectral graph convolution formula:
$$H^{(l+1)} = \sigma\left(\tilde{D}^{-\frac{1}{2}} \tilde{A} \tilde{D}^{-\frac{1}{2}} H^{(l)} W^{(l)}\right)$$
where $\tilde{A} = A + I_N$ (Adjacency matrix with self-loops) and $\tilde{D}$ is the diagonal degree matrix.

In [ ]:
class GraphConvolutionLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=True)
        
    def forward(self, x, adj):
        # Add self loops
        num_nodes = adj.size(0)
        adj_tilde = adj + torch.eye(num_nodes)
        
        # Degree matrix
        deg = torch.sum(adj_tilde, dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
        D_tilde = torch.diag(deg_inv_sqrt)
        
        # Normalized symmetric adjacency matrix
        norm_adj = torch.matmul(torch.matmul(D_tilde, adj_tilde), D_tilde)
        
        # Graph convolution operation
        support = self.linear(x)
        output = torch.matmul(norm_adj, support)
        return F.relu(output)

# Instantiate and test GCN on 6 nodes
num_nodes = 6
in_features = 8
out_features = 4

torch.manual_seed(42)
node_features = torch.randn(num_nodes, in_features)
# Adjacency matrix for 6 connected nodes
adj = torch.tensor([
    [0, 1, 1, 0, 0, 0],
    [1, 0, 1, 1, 0, 0],
    [1, 1, 0, 0, 1, 0],
    [0, 1, 0, 0, 1, 1],
    [0, 0, 1, 1, 0, 1],
    [0, 0, 0, 1, 1, 0]
], dtype=torch.float32)

gcn_layer = GraphConvolutionLayer(in_features, out_features)
out_embeddings = gcn_layer(node_features, adj)

print("=== GCN Forward Pass Output ===")
print(f"Input Node Features Shape : {node_features.shape}")
print(f"Output Embeddings Shape   : {out_embeddings.shape}")
print(f"Sample Node 0 Embedding   : {out_embeddings[0].detach().numpy().round(4)}")